In [ ]:
# Week 4 — Statistical Analysis and Correlation Study
# Waste Classification Dataset (Image-based)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# === 1. SETUP PATH ===
DATASET_DIR = r"C:\Users\user\Desktop\jupyter notebook\garbage\garbage-dataset"

# Verify
print("📂 Dataset path exists:", os.path.exists(DATASET_DIR))
classes = sorted(os.listdir(DATASET_DIR))
print("Classes found:", classes)

# === 2. SAMPLE IMAGE FEATURES EXTRACTION ===
# We'll compute basic image statistics per file: mean R,G,B and brightness

def image_stats(img_path):
    img = Image.open(img_path).convert("RGB")
    arr = np.array(img) / 255.0
    r_mean = arr[:, :, 0].mean()
    g_mean = arr[:, :, 1].mean()
    b_mean = arr[:, :, 2].mean()
    brightness = arr.mean()
    return r_mean, g_mean, b_mean, brightness

records = []
for cls in classes:
    cls_dir = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(cls_dir):
        continue
    files = [f for f in os.listdir(cls_dir) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
    for f in files[:80]:  # limit sample to 80 per class to keep it fast
        fpath = os.path.join(cls_dir, f)
        r, g, b, bright = image_stats(fpath)
        records.append({
            "class": cls,
            "r_mean": r,
            "g_mean": g,
            "b_mean": b,
            "brightness": bright
        })

df = pd.DataFrame(records)
print("\n✅ Sample dataframe created:", df.shape)
df.head()


In [ ]:
# === 3. DESCRIPTIVE STATISTICS ===
desc = df.describe()
print("\n📊 Descriptive Statistics:\n", desc)

# Visualize distributions
df[['r_mean', 'g_mean', 'b_mean', 'brightness']].hist(figsize=(10,6), bins=20, color='skyblue')
plt.suptitle("Distribution of Basic Image Statistics")
plt.show()


In [ ]:
# === 4. CORRELATION ANALYSIS ===
corr = df[['r_mean', 'g_mean', 'b_mean', 'brightness']].corr()
print("\n🔗 Correlation Matrix:\n", corr)

plt.figure(figsize=(6,5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Image Features")
plt.show()

# Example interpretation: brightness vs color channels
corr_target = corr['brightness'].drop('brightness').abs().sort_values(ascending=False)
top3 = corr_target.head(3)
print("\n💡 Top 3 features most correlated with Brightness:\n", top3)


In [ ]:
# === 6. SHORT SUMMARY REPORT ===
summary = f"""
📈 Week 4 Statistical Summary
-----------------------------
Total images analyzed: {len(df)}

Top 3 features correlated with brightness:
{top3.to_string()}

Observation:
Images with higher 'r_mean' and 'g_mean' values generally correspond
to brighter classes (e.g., paper/plastic), whereas darker classes
like batteries show lower brightness and blue channel dominance.
"""

with open("Week4_Report.txt", "w") as f:
    f.write(summary)

print(summary)
print("\n✅ Saved Week4_Report.txt for GitHub upload.")
